<a href="https://colab.research.google.com/github/titan-spyer/PRODIGY_GA_01/blob/main/Text_Generation_with_GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fine-tuning GPT-2 for Text Generation

This notebook will guide you through the process of fine-tuning a GPT-2 model on a custom dataset for text generation. We'll cover environment setup, data preparation, model training, and text generation.

### 1. Environment Setup

First, we need to install the `transformers` library, which provides access to pre-trained models like GPT-2 and tools for fine-tuning.

In [ ]:
pip install transformers datasets accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

### 2. Data Preparation

To fine-tune GPT-2, you need a custom dataset. This dataset should ideally be a collection of text in a plain text file, with each example separated by a specific token (e.g., a newline character or a special separator token you define). The more data you have, the better the fine-tuning results will typically be.

Here's how you can prepare and load your dataset:

1.  **Create a text file**: Put all your training text into a single `.txt` file.
2.  **Upload the file**: Upload this file to your Colab environment (e.g., to `/content/my_dataset.txt`).



In [ ]:
from datasets import Dataset
import os

def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"DEBUG: load_dataset read {len(lines)} raw lines from {file_path}")
    examples = [line.strip() for line in lines if line.strip()]
    return Dataset.from_dict({'text': examples})

dataset_path = 'my_dataset.txt'

# Ensure the dataset file is always created or overwritten correctly
if os.path.exists(dataset_path):
    os.remove(dataset_path)
    print(f"Removed existing dummy dataset file at {dataset_path}")

part1 = "This is the first sentence of my custom text dataset. It contains multiple sentences to train the GPT-2 model. The model will learn to generate text similar in style and content to this data."
part2 = "Here is another paragraph, providing more context and examples for the model to learn from. Fine-tuning on diverse and relevant data is crucial for good performance."
all_lines = []
for i in range(200): # Create 200 repetitions of each part, totaling 400 distinct examples
    all_lines.append(part1 + f" (example {2*i})") # Add unique identifier to ensure distinct examples
    all_lines.append(part2 + f" (example {2*i+1})")

print(f"DEBUG: Prepared {len(all_lines)} examples for writing.")
long_dummy_text = "\n".join(all_lines) # Join with single newline to create multiple lines/examples
print(f"DEBUG: long_dummy_text has {len(long_dummy_text.splitlines())} lines before writing.")

with open(dataset_path, 'w', encoding='utf-8') as f:
    f.write(long_dummy_text)

print(f"Created a longer dummy dataset file at {dataset_path} with {len(all_lines)} examples.")
print(f"DEBUG: my_dataset.txt exists: {os.path.exists(dataset_path)}")

raw_dataset = load_dataset(dataset_path)
print(f"Dataset loaded with {len(raw_dataset)} examples.")
# Print first 200 characters of the first example (if exists) or a message if empty
if len(raw_dataset) > 0:
    print(raw_dataset[0]['text'][:200])
else:
    print("Raw dataset is empty after loading.")

### 3. Tokenization

Now, we need to tokenize the text data. Tokenization is the process of converting raw text into numerical sequences that a model can understand. GPT-2 uses a Byte-Pair Encoding (BPE) tokenizer. We'll load the tokenizer associated with the pre-trained `gpt2` model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# GPT-2 does not have a padding token by default, which is needed for batching.
# We set it to the eos_token, which is common practice for GPT-2 fine-tuning for causal language modeling.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=1024)

tokenized_datasets = raw_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=1, # Adjust as needed based on your system's capabilities
    remove_columns=raw_dataset.column_names
)

print(f"Tokenized dataset: {tokenized_datasets}")
print(f"First tokenized example (input_ids): {tokenized_datasets[0]['input_ids'][:20]}")

### 4. Grouping and Preparing for Training

For causal language modeling, it's efficient to concatenate all texts and then split them into fixed-size chunks (`block_size`). This ensures that the model always receives sequences of a consistent length.

We will define a `block_size` (e.g., 128, 512, or 1024) and then process the tokenized dataset into these blocks. The `labels` for causal language modeling are typically the `input_ids` shifted by one position, meaning the model predicts the next token in the sequence.

In [ ]:
block_size = 128 # You can adjust this based on your computational resources and desired sequence length

# Adding a debug print to see the actual length of the tokenized input
if len(tokenized_datasets) > 0:
    # Concatenate all input_ids from the first batch to get the effective length for grouping
    # Note: tokenized_datasets is not batched at this point, so sum(tokenized_datasets['input_ids'], []) concatenates all examples
    print(f"DEBUG: Total length of tokenized_datasets examples before grouping: {len(sum(tokenized_datasets['input_ids'], []))}")
else:
    print("DEBUG: tokenized_datasets is empty before grouping.")

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=1, # Adjust as needed
)

print(f"Language Model ready dataset: {lm_datasets}")
if len(lm_datasets) > 0:
    print(f"First example input_ids length: {len(lm_datasets[0]['input_ids'])}")
    print(f"First example labels length: {len(lm_datasets[0]['labels'])}")
else:
    print("lm_datasets is empty after grouping.")

### 5. Loading the Pre-trained GPT-2 Model

We'll load the `gpt2` model using `AutoModelForCausalLM`, which is designed for causal language modeling (predicting the next token in a sequence).

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2")

print("GPT-2 model loaded successfully!")

### 6. Fine-tuning the Model

To fine-tune the model, we'll use the `Trainer` API from the `transformers` library. This requires defining `TrainingArguments` which specify hyper-parameters for training, and a `DataCollatorForLanguageModeling` to handle batching and dynamic padding if necessary.

In [ ]:
# Initialize Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False # mlm=False for causal language modeling
)

# Define Training Arguments
training_args = TrainingArguments(
    output_dir="./gpt2-fine-tuned", # Directory where the model predictions and checkpoints will be written
    # overwrite_output_dir=True, # Removed as it caused a TypeError
    num_train_epochs=3,           # Total number of training epochs
    per_device_train_batch_size=4, # Batch size per GPU/TPU core/CPU for training
    save_steps=10_000,            # Save checkpoint every X updates steps
    save_total_limit=2,           # Limit the total amount of checkpoints. Deletes older checkpoints.
    prediction_loss_only=True,    # Only calculate loss, no predictions during evaluation
    logging_dir='./logs',         # Directory for storing logs
    logging_steps=500,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available() # Enable mixed precision training if CUDA is available
)

# Create the Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets, # Our prepared dataset
)

print("Trainer initialized. Starting training...")

# Train the model
trainer.train()

print("Training complete! Saving model...")

# Save the fine-tuned model and tokenizer
trainer.save_model("./gpt2-fine-tuned")
tokenizer.save_pretrained("./gpt2-fine-tuned")
print("Model and tokenizer saved to ./gpt2-fine-tuned")

### 7. Text Generation

Now that the model is fine-tuned, we can load it and use it to generate new text based on a given prompt. We'll load the fine-tuned model and tokenizer, and then use the `generate` method.

In [ ]:
# Load the fine-tuned model and tokenizer
fine_tuned_model_path = "./gpt2-fine-tuned"
fine_tuned_model = AutoModelForCausalLM.from_pretrained(fine_tuned_model_path)
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)

print(f"Fine-tuned model loaded from {fine_tuned_model_path}")

Let's generate some text! You can change the `prompt` variable to try different starting phrases.

In [ ]:
prompt = "This is the first sentence of my custom text dataset."

# Encode the prompt
input_ids = fine_tuned_tokenizer.encode(prompt, return_tensors='pt')

# Generate text
output = fine_tuned_model.generate(
    input_ids,
    max_length=100,  # Maximum length of the generated text
    num_return_sequences=1, # Number of sequences to generate
    no_repeat_ngram_size=2, # Avoid repeating n-grams
    do_sample=True, # Use sampling for more diverse text
    top_k=50, # Consider only the top 50 most likely tokens
    top_p=0.95, # Use nucleus sampling
    temperature=0.7, # Control randomness
    pad_token_id=fine_tuned_tokenizer.eos_token_id # Use EOS token for padding
)

# Decode and print the generated text
generated_text = fine_tuned_tokenizer.decode(output[0], skip_special_tokens=True)
print("\nGenerated Text:")
print(generated_text)

### 8. Conclusion

This notebook demonstrated the end-to-end process of fine-tuning GPT-2 for text generation on a custom dataset. We covered:

1.  **Environment Setup**: Installing necessary libraries.
2.  **Data Preparation**: Creating and loading a dummy dataset.
3.  **Tokenization**: Converting text into numerical input for the model.
4.  **Grouping for Training**: Preparing the data into fixed-size blocks.
5.  **Model Loading**: Loading the pre-trained GPT-2 model.
6.  **Fine-tuning**: Training the model on the custom dataset.
7.  **Text Generation**: Using the fine-tuned model to generate new text.

To further improve the model, you could:

*   Use a larger and more diverse custom dataset.
*   Experiment with different `TrainingArguments` (e.g., learning rate, batch size, number of epochs).
*   Try different GPT-2 model sizes (e.g., `gpt2-medium`, `gpt2-large`).
*   Implement evaluation metrics to quantitatively assess the generated text quality.